# Notebook 04 — Cellular Automata

**Module:** 15 — Simulation and Agent-Based Modeling  
**Tier:** 2 — Working competence  
**Notebook:** 4 of 12  
**Time estimate:** 70 minutes

> A cellular automaton (CA) updates a grid of discrete states using local rules
> applied simultaneously to all cells. Despite having no global coordinator and
> only local information, CAs produce complex emergent behaviour: gliders, spirals,
> propagating waves, and growth patterns that mirror biology.

---
## Step 1 — Motivation

Conway's Game of Life (1970) showed that a 2-state, 4-rule grid automaton can
produce Turing-complete computation. The Greenberg-Hastings model captures the
propagating waves of cardiac excitation — and the spiral waves that cause
ventricular fibrillation. The forest fire CA models epidemic spread and ecological
patch dynamics. CAs are the simplest models that demonstrate how local rules
generate global order without a blueprint.

---
## Step 2 — Intuition

**1D cellular automaton (Wolfram):**
- N cells, each either 0 (dead) or 1 (alive)
- At each step, each cell's new state is determined by its own state and
  its two neighbours — 3 cells, 2 states = 2³=8 possible neighbourhoods
- A 1D CA rule is a lookup table: 8 inputs → 8 outputs = 2⁸=256 possible rules
- Rule 110 is universal (Turing-complete)

**Conway's Game of Life (2D):**
- 2 states (0/1), 8-neighbourhood (Moore)
- Birth: a dead cell with exactly 3 live neighbours becomes alive
- Survival: a live cell with 2 or 3 live neighbours stays alive
- Death: otherwise
- Simple rules → gliders, oscillators, still lifes, universal computers

**Greenberg-Hastings (excitable media):**
- 3 states: 0 (quiescent), 1 (excited/refractory)
- Excited cells transition to refractory (can't be re-excited — refractory period)
- Quiescent cells become excited if they have ≥1 excited neighbour
- Produces propagating waves with guaranteed directionality

---
## Step 3 — Biological Background

**Cardiac excitable media:** Heart muscle cells are excitable — they fire an
action potential and then enter a refractory period. Normal heartbeat propagates
as a coordinated wave. In ventricular fibrillation, spiral waves break into
chaotic small waves — the Greenberg-Hastings model captures this transition.

**Forest fire / epidemic analogy:** The forest fire CA (tree → burning → empty)
is isomorphic to SIR (S → I → R). It shows the same epidemic threshold: a fire
can propagate only if tree density exceeds a critical value (analogous to
R₀ > 1).

**Dictyostelium cAMP waves:** slime mold aggregation during starvation is driven
by spiral cAMP waves — a biological excitable medium strikingly similar to the
Greenberg-Hastings pattern.

**Criticality and the edge of chaos:** Wolfram's classification (Class I-IV)
and Langton's λ parameter: biological CAs tend to operate near the phase
transition between order (Class II) and chaos (Class III) — the "edge of chaos"
where information processing is maximized.

---
## Step 4 — Mathematical Explanation

**1D CA rule encoding:**
A rule number $R$ in binary gives the 8-bit lookup table:
$$\text{new}(L, C, R) = \text{bit}_{\text{idx}}(R), \quad \text{idx} = 4L + 2C + R$$
e.g., Rule 30: binary 00011110 → bit 1 = new state for neighbourhood 001, etc.

**Game of Life update (vectorized):**
$$N_{i,j} = \sum_{(\delta i, \delta j) \in \text{Moore}} \text{grid}[i+\delta i, j+\delta j]$$
$$\text{grid}_{t+1}[i,j] = \begin{cases}
1 & \text{if } \text{grid}_t = 1 \text{ and } N \in \{2,3\} \\
1 & \text{if } \text{grid}_t = 0 \text{ and } N = 3 \\
0 & \text{otherwise}
\end{cases}$$

Using `scipy.signal.convolve2d` with a 3×3 kernel of ones (minus centre) for N
is the fastest pure-numpy implementation.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
from scipy.signal import convolve2d

rng = np.random.default_rng(42)

# ---- 1D Cellular Automaton ----
def ca1d_step(row, rule_number):
    """One step of 1D CA with given Wolfram rule number."""
    rule = np.array([(rule_number >> i) & 1 for i in range(8)], dtype=np.uint8)
    n = len(row)
    new_row = np.zeros(n, dtype=np.uint8)
    for i in range(n):
        L = row[(i - 1) % n]
        C = row[i]
        R = row[(i + 1) % n]
        idx = 4*L + 2*C + R
        new_row[i] = rule[idx]
    return new_row

def run_ca1d(rule_number, n_cells=150, n_steps=75, seed_center=True):
    history = np.zeros((n_steps, n_cells), dtype=np.uint8)
    if seed_center:
        history[0, n_cells // 2] = 1
    else:
        history[0] = rng.integers(0, 2, n_cells)
    for t in range(1, n_steps):
        history[t] = ca1d_step(history[t-1], rule_number)
    return history

ca30  = run_ca1d(30,  seed_center=True)
ca110 = run_ca1d(110, seed_center=True)
ca90  = run_ca1d(90,  seed_center=True)

# ---- Conway's Game of Life ----
NEIGH_KERNEL = np.ones((3, 3), dtype=np.int32)
NEIGH_KERNEL[1, 1] = 0  # exclude self

def gol_step(grid):
    """One step of Conway's Game of Life."""
    N = convolve2d(grid, NEIGH_KERNEL, mode='same', boundary='wrap')
    birth    = (grid == 0) & (N == 3)
    survival = (grid == 1) & ((N == 2) | (N == 3))
    return (birth | survival).astype(np.int32)

def run_gol(grid, n_steps):
    history = [grid.copy()]
    for _ in range(n_steps):
        grid = gol_step(grid)
        history.append(grid.copy())
    return history

# Canonical patterns
def make_gol_grid(N=60):
    G = np.zeros((N, N), dtype=np.int32)
    # Glider (top-left)
    glider = np.array([[0,1,0],[0,0,1],[1,1,1]], dtype=np.int32)
    G[2:5, 2:5] = glider
    # Blinker (middle)
    G[30, 28:31] = 1
    # Still life: block
    G[45:47, 45:47] = 1
    return G

gol_init = make_gol_grid(60)
gol_history = run_gol(gol_init, n_steps=30)

# Count living cells
living_counts = [g.sum() for g in gol_history]
print(f'Game of Life: initial={living_counts[0]}, after 30 steps={living_counts[-1]}')

# ---- Greenberg-Hastings (excitable medium) ----
def gh_step(grid, n_states=8):
    """
    Greenberg-Hastings excitable medium.
    State 0 = quiescent, 1 = excited, 2..n_states-1 = refractory.
    """
    new_grid = grid.copy()
    # Refractory → advance (increment or → 0)
    refract = (grid >= 2)
    new_grid[refract] = (grid[refract] + 1) % n_states
    # Quiescent → excited if any excited (state=1) neighbour
    quiescent = (grid == 0)
    n_excited = convolve2d((grid == 1).astype(int), np.ones((3,3)), mode='same', boundary='wrap')
    new_grid[quiescent & (n_excited >= 1)] = 1
    # Excited → refractory 2
    new_grid[grid == 1] = 2
    return new_grid

def run_gh(N=80, n_steps=60, n_states=8):
    G = np.zeros((N, N), dtype=int)
    # Seed a spiral by placing excited cells in an arc
    cx, cy = N//2, N//2
    r = 10
    for theta in np.linspace(0, np.pi, 40):
        x = int(cx + r * np.cos(theta))
        y = int(cy + r * np.sin(theta))
        if 0 <= x < N and 0 <= y < N:
            G[x, y] = 1
    for theta in np.linspace(0, np.pi/2, 20):
        x = int(cx + (r+1) * np.cos(theta))
        y = int(cy + (r+1) * np.sin(theta))
        if 0 <= x < N and 0 <= y < N:
            G[x, y] = 2
    history_gh = [G.copy()]
    for _ in range(n_steps):
        G = gh_step(G, n_states)
        history_gh.append(G.copy())
    return history_gh

gh_history = run_gh()
print(f'Greenberg-Hastings: {len(gh_history)} steps run')

In [ ]:
# Step 7 — Visualization
fig, axes = plt.subplots(2, 3, figsize=(18, 11))

# Panel A: 1D CA rules
ax = axes[0, 0]
ca_plot = np.vstack([ca30, ca110, ca90])
ax.imshow(ca_plot, cmap='binary', aspect='auto', interpolation='none')
ax.axhline(75,  color='tomato', lw=1.5, ls='--')
ax.axhline(150, color='steelblue', lw=1.5, ls='--')
ax.set_yticks([37, 112, 187])
ax.set_yticklabels(['Rule 30 (chaotic)', 'Rule 110 (complex)', 'Rule 90 (fractal)'])
ax.set_xlabel('Cell position'); ax.set_title('A. 1D Cellular Automata\n(75 steps each, single-cell seed)')

# Panel B: Game of Life — initial state
ax = axes[0, 1]
ax.imshow(gol_history[0], cmap='binary', interpolation='none')
ax.set_title('B. Game of Life — initial state\n(glider + blinker + block)')
ax.axis('off')

# Panel C: Game of Life — after 30 steps
ax = axes[0, 2]
ax.imshow(gol_history[30], cmap='binary', interpolation='none')
ax.set_title('C. Game of Life — after 30 steps\n(glider has moved, blinker oscillated)')
ax.axis('off')

# Panel D: Greenberg-Hastings wave
cmap_gh = plt.get_cmap('tab10')
ax = axes[1, 0]
ax.imshow(gh_history[40], cmap='hot', interpolation='bilinear', vmin=0, vmax=8)
ax.set_title('D. Greenberg-Hastings\n(excitable medium, step 40)')
ax.axis('off')

# Panel E: Living cells over time (Game of Life)
ax = axes[1, 1]
ax.plot(range(len(living_counts)), living_counts, 'steelblue', lw=2)
ax.set_xlabel('Step'); ax.set_ylabel('Living cells')
ax.set_title('E. Game of Life: living cell count\n(glider + blinker + block)')

# Panel F: Forest fire CA
def run_forest_fire(N=100, n_steps=80, p_tree=0.7, p_grow=0.005, p_fire=0.001, seed=42):
    """
    Forest fire CA: 0=empty, 1=tree, 2=burning.
    Tree can spontaneously ignite (lightning p_fire per step).
    """
    rng_ff = np.random.default_rng(seed)
    G = (rng_ff.random((N, N)) < p_tree).astype(int)
    history = [G.copy()]
    for _ in range(n_steps):
        new_G = G.copy()
        # Burning → empty
        new_G[G == 2] = 0
        # Tree near fire → burns
        near_fire = convolve2d((G == 2).astype(int), np.ones((3,3)), mode='same', boundary='wrap') > 0
        new_G[(G == 1) & near_fire] = 2
        # Empty → tree growth
        new_G[(G == 0) & (rng_ff.random((N, N)) < p_grow)] = 1
        # Lightning
        new_G[(G == 1) & (rng_ff.random((N, N)) < p_fire)] = 2
        G = new_G
        history.append(G.copy())
    return history

ff_history = run_forest_fire()
cmap_ff = mcolors.ListedColormap(['#f5deb3', '#228b22', '#ff4500'])  # sand, forest, fire
ax = axes[1, 2]
ax.imshow(ff_history[-1], cmap=cmap_ff, vmin=0, vmax=2, interpolation='none')
ax.set_title('F. Forest fire CA\n(green=tree, red=burning, tan=empty)')
ax.axis('off')

plt.suptitle('Module 15 NB04: Cellular Automata', fontsize=12, fontweight='bold')
plt.tight_layout()
plt.savefig('cellular_automata.png', dpi=150, bbox_inches='tight')
plt.show()

---
## Step 8 — Exercises

1. Implement a glider gun in Game of Life (seed the Gosper glider gun pattern
   from the known coordinate list). Show that it produces gliders indefinitely.
2. Show the forest fire CA exhibits a phase transition: sweep tree density
   p_tree from 0.3 to 0.9 and plot the fraction of the grid that eventually
   burns. Identify the critical p_tree.
3. Modify the Greenberg-Hastings model to have 3 refractory states instead of 6.
   What happens to the wave pattern? Why? (Hint: think about the refractory
   period length relative to wave propagation speed.)

---
## Step 10 — Quiz

1. What are Wolfram's four classes of 1D CA behaviour? Which class does
   Rule 30 belong to, and which does Rule 110?
2. In the Game of Life, what are the four update rules? Why does a live cell
   with 4 neighbours die?
3. The Greenberg-Hastings model captures the refractory period of cardiac muscle.
   Why is a refractory period essential for a propagating wave to have a
   definite direction rather than spreading in all directions from the origin?
4. How is the forest fire CA isomorphic to the SIR epidemic model?
   Map the three forest states to S, I, R compartments.

---
## Step 12 — Reflection

> *[In one paragraph: Conway's Game of Life has only 4 rules and 2 states, yet
> it is Turing-complete. What does this tell us about the relationship between
> the complexity of rules and the complexity of emergent behaviour? Apply this
> thinking to biological development: why is it plausible that complex body plans
> could arise from a small set of molecular signaling rules?]*

---
*Next: `05_abm_fundamentals.ipynb`*